# IOAI — 2025 Team Selection Day1 Cv Image Restoration (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2025-team-selection-day1-cv-image-restoration'
for f in ['train_clean.npy','train_filtered.npy','val_filtered.npy']:
    if not os.path.exists(f): os.system(f'wget -q {BASE}/{f}')
print('데이터 준비:', [f for f in ['train_clean.npy','train_filtered.npy','val_filtered.npy'] if os.path.exists(f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Day 1 — Image Restoration · 모범답안 (Reference Solution)

> **Kazakhstan – Team Selection 2025 · Day 1**

Faithful reproduction of the strong approach (Batyr Yerdenov's TST solution): a **U-Net with Squeeze-and-Excitation blocks** that is **conditioned on the detected 2×2 filter** (one-hot), trained with **MSE + L1**. Two key boosts over the baseline: (1) the network is told *which* filter was applied, and (2) since the kept channel equals the original exactly, we **paste back the known channels** after prediction — the model only has to inpaint the missing ones. Saves `val_pred.npy` for PSNR grading.

In [ ]:
import os, numpy as np, torch
def _find(f):
    for b in [".","..","../..","/home/yhpark/ioai/problems/Kazakhstan-2025/Team-Selection/Day1_CV_Image_Restoration"]:
        if os.path.exists(os.path.join(b,f)): return b
    return "."

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
DATA = _find("train_clean.npy")
Xtr_c = np.load(os.path.join(DATA,"train_clean.npy"))      # (N,128,128,3) uint8  (clean targets)
Xtr_f = np.load(os.path.join(DATA,"train_filtered.npy"))   # (N,128,128,3) uint8  (filtered inputs)
Xva_f = np.load(os.path.join(DATA,"val_filtered.npy"))     # (M,128,128,3) uint8  (val inputs)
print("train", Xtr_c.shape, "val", Xva_f.shape, "device", device)

def detect_filter(f):
    """Each 2x2 block keeps exactly one colour channel per pixel. Return per-position kept channel (4 ids 0..2)."""
    m = (f > 0)
    key = []
    for py in range(2):
        for px in range(2):
            key.append(int(m[py::2, px::2, :].reshape(-1, 3).mean(0).argmax()))
    return key  # e.g. [2,0,0,1]


In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# known-channel mask per image (1 where the channel was kept/known, 0 where it must be inpainted)
def known_mask(fimg):
    k = detect_filter(fimg); m = np.zeros((128,128,3), np.float32)
    for idx,(py,px) in enumerate([(0,0),(0,1),(1,0),(1,1)]):
        m[py::2, px::2, k[idx]] = 1.0
    fid = FILT2ID.get(tuple(k), 0)
    return m, fid

# enumerate the distinct filters present in the training set -> stable id table
FILT2ID = {}
for i in range(len(Xtr_f)):
    k = tuple(detect_filter(Xtr_f[i]))
    if k not in FILT2ID: FILT2ID[k] = len(FILT2ID)
NF = max(12, len(FILT2ID))
print("distinct filters:", len(FILT2ID), "-> one-hot dim", NF)

class DS(Dataset):
    def __init__(s, Xf, Xc=None):
        s.Xf=Xf; s.Xc=Xc
    def __len__(s): return len(s.Xf)
    def __getitem__(s, i):
        f=s.Xf[i]; m,fid=known_mask(f)
        x=torch.tensor(f.astype(np.float32)/255.).permute(2,0,1)
        mk=torch.tensor(m).permute(2,0,1)
        oh=torch.zeros(NF); oh[fid]=1.
        if s.Xc is None: return x, mk, oh
        y=torch.tensor(s.Xc[i].astype(np.float32)/255.).permute(2,0,1)
        return x, mk, oh, y
tr=DataLoader(DS(Xtr_f,Xtr_c), batch_size=32, shuffle=True)


## U-Net + Squeeze-Excitation + filter conditioning

In [ ]:
class SE(nn.Module):
    def __init__(s,c,r=8):
        super().__init__(); s.fc=nn.Sequential(nn.Linear(c,c//r),nn.ReLU(True),nn.Linear(c//r,c),nn.Sigmoid())
    def forward(s,x):
        w=s.fc(x.mean((2,3))).unsqueeze(-1).unsqueeze(-1); return x*w
def cbr(i,o): return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                                   nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class SEUNet(nn.Module):
    def __init__(s, nf):
        super().__init__()
        s.d1=cbr(3,32); s.d2=cbr(32,64); s.d3=cbr(64,128); s.pool=nn.MaxPool2d(2)
        s.ffc=nn.Linear(nf,128); s.bott=cbr(128+1,256); s.se=SE(256)
        s.u3=nn.ConvTranspose2d(256,128,2,2); s.c3=cbr(256,128); s.s3=SE(128)
        s.u2=nn.ConvTranspose2d(128,64,2,2);  s.c2=cbr(128,64);  s.s2=SE(64)
        s.u1=nn.ConvTranspose2d(64,32,2,2);   s.c1=cbr(64,32)
        s.out=nn.Conv2d(32,3,1)
    def forward(s,x,oh):
        e1=s.d1(x); e2=s.d2(s.pool(e1)); e3=s.d3(s.pool(e2)); p=s.pool(e3)
        f=s.ffc(oh).unsqueeze(-1).unsqueeze(-1).mean(1,keepdim=True).expand(-1,1,p.shape[2],p.shape[3])
        b=s.se(s.bott(torch.cat([p,f],1)))
        d=s.s3(s.c3(torch.cat([s.u3(b),e3],1))); d=s.s2(s.c2(torch.cat([s.u2(d),e2],1))); d=s.c1(torch.cat([s.u1(d),e1],1))
        return torch.sigmoid(s.out(d))
model=SEUNet(NF).to(device); opt=torch.optim.Adam(model.parameters(),1e-3)


## Train (MSE + L1)

In [ ]:
EPOCHS=15
for ep in range(EPOCHS):
    model.train(); tot=0
    for x,mk,oh,y in tr:
        x,mk,oh,y=x.to(device),mk.to(device),oh.to(device),y.to(device)
        opt.zero_grad(); out=model(x,oh)
        loss=F.mse_loss(out,y)+0.2*F.l1_loss(out,y); loss.backward(); opt.step()
        tot+=loss.item()*len(x)
    print(f"epoch {ep+1}/{EPOCHS}  loss={tot/len(tr.dataset):.4f}")


## Predict + paste known channels → `val_pred.npy`

In [ ]:
model.eval()
va=DataLoader(DS(Xva_f), batch_size=64)
out_imgs=[]
with torch.no_grad():
    bi=0
    for x,mk,oh in va:
        x,mk,oh=x.to(device),mk.to(device),oh.to(device)
        pred=model(x,oh).clamp(0,1)
        pred=pred*(1-mk)+x*mk           # keep exact known channels, inpaint the rest
        out_imgs.append((pred.cpu().numpy()*255).round().astype(np.uint8)); bi+=1
val_pred=np.concatenate(out_imgs).transpose(0,2,3,1)
np.save("val_pred.npy", val_pred)
print("saved val_pred.npy", val_pred.shape)


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['val_pred.npy']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)